# Task 1 - OpenAI Setup and Basic Prompt

In this task, I configured the OpenAI API key and tested the OpenAI chat model by sending a simple prompt.

In [1]:
# Install required libraries
!pip install -q langchain langchain-openai openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.3/558.3 kB 11.3 MB/s eta 0:00:00


In [2]:
from google.colab import userdata
import os
# Loading the API key from Colab Secrets
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
print("OpenAI API Key loaded successfully!")

OpenAI API Key loaded successfully!


In [3]:
from langchain_openai import ChatOpenAI
# Loading OpenAI model
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
print("Model Loaded")

Model Loaded


In [ ]:
# Simple prompt

response = llm.invoke("Explain Retrieval-Augmented Generation in one sentence.")
print("Response:\n")
print(response.content)

In [5]:
"""The OpenAI model responded successfully, confirming that the API key and chat model are working correctly,
hence task 1 is completed """

'The OpenAI model responded successfully, confirming that the API key and chat model are working correctly, \nhence task 1 is completed '

# Task 2 - Wikipedia Retriever

In this task, I used LangChain's `WikipediaRetriever` to retrieve information from Wikipedia based on a user query.

In [ ]:
 !pip install -q wikipedia langchain-community
from langchain_community.retrievers import WikipediaRetriever
print("Wikipedia Retriever Imported")

In [8]:
# Create retriever
retriever = WikipediaRetriever(top_k_results=2)
print("Retriever Created")

query = "Retrieval-Augmented Generation"
docs = retriever.invoke(query)
print("Number of documents retrieved:", len(docs))

Retriever Created
Number of documents retrieved: 2


In [9]:
#Display the retrieved content
for i, doc in enumerate(docs, start=1):
    print(f"\nDocument {i}")
    print("-" * 50)
    print(doc.page_content[:700])   # Print first 700 characters


Document 1
--------------------------------------------------
Retrieval-augmented generation (RAG) is a technique that enables large language models (LLMs) to retrieve and incorporate new information from external data sources. With RAG, LLMs first refer to a specified set of documents, then respond to user queries. These documents supplement information from the LLM's pre-existing training data. This allows LLMs to use domain-specific and/or updated information that is not available in the training data. For example, this enables LLM-based chatbots to access internal company data or generate responses based on authoritative sources. The technique was first proposed in 2020 and has since become a widely adopted approach in modern AI systems.
RAG improv

Document 2
--------------------------------------------------
A vector database, vector store or vector search engine is a database that stores and retrieves embeddings of data in vector space. Vector databases typically implement appr

### Observation

The Wikipedia Retriever successfully fetched relevant documents based on the given query. The retrieved content can be used as external knowledge for a RAG system.

# Task 3 - Vector Store Retriever

In this task, I converted the retrieved Wikipedia documents into embeddings using OpenAI, stored them in a FAISS vector database, and performed similarity search using a retriever.

In [10]:
!pip install -q faiss-cpu
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
print("Required libraries imported")

#creating the embeddings
embeddings = OpenAIEmbeddings()
print("Embedding model loaded")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 59.4 MB/s eta 0:00:00
Required libraries imported
Embedding model loaded


In [ ]:
#Creating the FAISS Vector Store
vector_store = FAISS.from_documents(docs, embeddings)
print("FAISS vector store created")

#crete the retriever
retriever = vector_store.as_retriever()
print("Retriever created")

#performing the similarity search
question = "What is Retrieval-Augmented Generation?"
results = retriever.invoke(question)
print("Retrieved documents:", len(results))

In [ ]:
#finally displaying the results
for i, doc in enumerate(results, start=1):
    print(f"\nResult {i}")
    print("-" * 50)
    print(doc.page_content[:700])

### Observation

The retrieved Wikipedia documents were converted into vector embeddings and stored in FAISS. When a question was asked, the retriever successfully returned the most relevant documents based on semantic similarity.

# Task 4 - Maximal Marginal Relevance (MMR) Retriever

In this task, I compared the normal similarity search with Maximal Marginal Relevance (MMR) retrieval. MMR helps retrieve documents that are both relevant and diverse.

In [ ]:
# Normal Similarity Retriever
similarity_retriever = vector_store.as_retriever(search_type="similarity",search_kwargs={"k": 3})
# MMR Retriever
mmr_retriever = vector_store.as_retriever(search_type="mmr",search_kwargs={"k": 3, "fetch_k": 6})
query = "What are the advantages of Retrieval-Augmented Generation?"
print("Query:", query)

# Retrieve using similarity search
similarity_docs = similarity_retriever.invoke(query)
print("\n===== Similarity Search Results =====")
for i, doc in enumerate(similarity_docs, 1):
    print(f"\nDocument {i}")
    print(doc.page_content[:300])

# Retrieve using MMR
mmr_docs = mmr_retriever.invoke(query)
print("\n===== MMR Search Results =====")
for i, doc in enumerate(mmr_docs, 1):
    print(f"\nDocument {i}")
    print(doc.page_content[:300])

print("\nTask 4 Completed Successfully")

### Observation

Both retrievers returned relevant documents. The similarity retriever focused only on the most similar chunks, while the MMR retriever selected documents that were relevant but also more diverse, reducing repeated information.

# Task 5 - Multi-Query Retriever

In this task, I used LangChain's MultiQueryRetriever to generate multiple versions of the same query. This helps retrieve more relevant documents from the vector database.

In [ ]:
#import the libraries
!pip install --upgrade langchain
from langchain.retrievers.multi_query import MultiQueryRetriever

In [ ]:
# Create MultiQuery Retriever
multi_retriever = MultiQueryRetriever.from_llm(retriever=vector_store.as_retriever(),llm=llm)

print("MultiQuery Retriever Created")
query = "What are the benefits of Retrieval-Augmented Generation?"
print("\nOriginal Query:")
print(query)

# Retrieve documents
docs = multi_retriever.invoke(query)
print("\nNumber of documents retrieved:", len(docs))
print("\nRetrieved Documents:\n")
for i, doc in enumerate(docs, 1):
    print(f"Document {i}")
    print("-" * 50)
    print(doc.page_content[:300])
    print()

### Observation

The MultiQuery Retriever generated multiple versions of the original query using the LLM. This helped retrieve additional relevant documents that may not have been found using a single query.

# Task 6 - Contextual Compression Retriever

In this task, I used a Contextual Compression Retriever to remove unnecessary information from the retrieved documents and keep only the relevant content for answering the user's query.

In [ ]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor

In [ ]:
# Base retriever
base_retriever = vector_store.as_retriever(search_kwargs={"k": 3})

# Create compressor
compressor = LLMChainExtractor.from_llm(llm)

# Create contextual compression retriever
compression_retriever = ContextualCompressionRetriever(base_compressor=compressor,base_retriever=base_retriever)
query = "What are the advantages of Retrieval-Augmented Generation?"
print("Query:", query)

# Before compression
before_docs = base_retriever.invoke(query)
print("\n========== Before Compression ==========")
for i, doc in enumerate(before_docs, 1):
    print(f"\nDocument {i}")
    print("-" * 50)
    print(doc.page_content[:400])

# After compression
after_docs = compression_retriever.invoke(query)
print("\n========== After Compression ==========")
for i, doc in enumerate(after_docs, 1):
    print(f"\nDocument {i}")
    print("-" * 50)
    print(doc.page_content)
print("\nTask 6 Completed Successfully")

### Observation

The Contextual Compression Retriever filtered the retrieved documents and kept only the information relevant to the user's query. Compared to the original retrieval, the compressed output was shorter, more focused, and easier for the LLM to use when generating answers.

In [ ]:
"""# Task 7 - Load YouTube Content

In this task, I loaded the transcript of a YouTube video using `YoutubeLoader` and split it into smaller chunks using a text splitter."""

from langchain_community.document_loaders import YoutubeLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

YouTube Video Used by me is this:
https://www.youtube.com/watch?v=aircAruvnKk

In [ ]:
# YouTube video URL
video_url = "https://www.youtube.com/watch?v=aircAruvnKk"
# Load transcript
loader = YoutubeLoader.from_youtube_url(video_url,add_video_info=True)
documents = loader.load()

print("Transcript loaded successfully")
print("Number of documents:", len(documents))

# Split transcript
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50)
chunks = text_splitter.split_documents(documents)

print("Transcript split into chunks")
print("Total chunks:", len(chunks))

print("\nFirst Chunk:\n")
print(chunks[0].page_content[:500])

print("\nTask 7 Completed Successfully")

# Task 8 - Build Vector Store for YouTube Content

In this task, I created embeddings for the YouTube transcript chunks, stored them in a FAISS vector database, and created a retriever for semantic search.

In [ ]:
# Create embeddings
embeddings = OpenAIEmbeddings()

print("Embedding model loaded")

# Create FAISS vector store
youtube_vector_store = FAISS.from_documents(chunks, embeddings)
print("Vector store created")

# Create retriever
youtube_retriever = youtube_vector_store.as_retriever(search_kwargs={"k": 3})
print("Retriever created")

# Test retrieval
query = "What is the main topic discussed in the video?"
results = youtube_retriever.invoke(query)
print("\nRetrieved Chunks:\n")
for i, doc in enumerate(results, 1):
    print(f"Chunk {i}")
    print("-" * 50)
    print(doc.page_content[:300])
    print()
print("Task 8 Completed Successfully")

### Observation

The transcript chunks were converted into embeddings and stored in FAISS. The retriever successfully returned the most relevant chunks based on the user's query.

In [ ]:
#task 9
from langchain.chains import ConversationalRetrievalChain
# Create RAG chatbot
qa_chain = ConversationalRetrievalChain.from_llm(llm=llm,retriever=youtube_retriever)

print("YouTube RAG Chatbot Ready")
chat_history = []

while True:
    question = input("\nAsk a question (type 'exit' to stop): ")
    if question.lower() == "exit":
        print("Chat ended.")
        break

    result = qa_chain.invoke({"question": question,"chat_history": chat_history})
    print("\nAnswer:")
    print(result["answer"])
    chat_history.append((question, result["answer"]))

The chatbot retrieves the most relevant transcript chunks and uses the OpenAI model to generate answers. The conversation history is maintained so follow-up questions are understood better.

In [ ]:
#tesing
test_questions = [
    "What is the video about?",
    "What are the important concepts discussed?",
    "Can you summarize the video?",
    "What example is explained in the video?",
    "Who is this video useful for?",
    "What is the capital of France?"]
chat_history = []

for question in test_questions:
    print("\nQuestion:", question)
    result = qa_chain.invoke({"question": question,"chat_history": chat_history})
    print("Answer:", result["answer"])
    chat_history.append((question, result["answer"]))

1. Difference between Retriever-Based RAG and Normal Prompting

Normal prompting uses only what the AI already knows. Retriever-Based RAG first finds useful information from documents or a database, then uses it to answer.

2. Why are Vector Stores Critical?

Vector stores save document embeddings and help find information based on meaning, not just exact words. This makes searching faster and more accurate.

3. When should MMR be used instead of Similarity Search?

Use similarity search when you want the closest matches. Use MMR when you want relevant results with less repeated information.

4. Benefits of Multi-Query Retrieval

It creates different versions of the same question to search better. This helps find information that one query might miss.

5. Importance of Contextual Compression

It removes unnecessary details and keeps only the important information. This saves tokens and gives clearer answers.